In [7]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
from ultralytics import YOLO

import cv2

In [5]:
model = YOLO("yolo11n-seg.yaml").load("yolo11n.pt")  # build from YAML and transfer weights
results = model.train(data="./dataset.yaml", epochs=100, imgsz=640)

Transferred 499/561 items from pretrained weights
Ultralytics 8.3.205  Python-3.13.5 torch-2.8.0+cpu CPU (12th Gen Intel Core i7-1260P)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=train6, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pat

In [6]:
# \runs\segment\train5\weights\last.pt

model = YOLO("./runs/segment/train6/weights/last.pt")
results = model.train(data="./dataset.yaml", epochs=100, imgsz=640)

Ultralytics 8.3.205  Python-3.13.5 torch-2.8.0+cpu CPU (12th Gen Intel Core i7-1260P)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=./runs/segment/train6/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train7, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, p

In [30]:
# \runs\segment\train6\weights\best.pt
model = YOLO("./runs/segment/train6/weights/best.pt")

img = cv2.imread("./dataset/train/images/2.png")

result = model.predict(source=img, imgsz=640, conf=0.25)
print(result)


0: 448x640 1 CAT, 1 DOG, 1 PERSON, 96.5ms
Speed: 3.0ms preprocess, 96.5ms inference, 4.2ms postprocess per image at shape (1, 3, 448, 640)
[ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: ultralytics.engine.results.Masks object
names: {0: 'CAT', 1: 'DOG', 2: 'PERSON'}
obb: None
orig_img: array([[[230, 206, 246],
        [230, 206, 246],
        [230, 206, 246],
        ...,
        [230, 206, 246],
        [230, 206, 246],
        [230, 206, 246]],

       [[230, 206, 246],
        [230, 206, 246],
        [230, 206, 246],
        ...,
        [230, 206, 246],
        [230, 206, 246],
        [230, 206, 246]],

       [[230, 206, 246],
        [230, 206, 246],
        [230, 206, 246],
        ...,
        [230, 206, 246],
        [230, 206, 246],
        [230, 206, 246]],

       ...,

       [[ 29,  25,  24],
        [ 29,  25,  24],
        [ 25,  23,  23],
        ...,
        [230, 206, 246],
       

In [48]:
mask = result[0].masks.xy
overlay = img.copy()
image = img.copy()

mbox = result[0]
# class_id = result[0].boxes.cls
# confidence = result[0].boxes.conf

for box in mbox.boxes:
  class_id = mbox.names[box.cls[0].item()]
  cords = box.xyxy[0].tolist()
  cords = [round(i) for i in cords]

  color = (0, 255, 0)  # Green color for the mask
  if (box.cls[0].item() == 0):  # Assuming class 0 is one category
      color = (0, 255, 0)  # Green for class 0
  elif (box.cls[0].item() == 1):  # Assuming class 1 is another category
      color = (255, 0, 0)  # Blue for class 1
  elif (box.cls[0].item() == 2):  # Assuming class
      color = (0, 0, 255)  # Red for class 2

  conf = round(box.conf[0].item(), 2)
  image = cv2.rectangle(image, (cords[0], cords[1]), (cords[2], cords[3]), color, 2)
  image = cv2.putText(image, f'{class_id} {conf}', (cords[0], cords[1]-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,0), 2)
  print(class_id, cords)

# Polygon Mask
for i in range(len(mask)):

    x = mask[i].astype(int)
    pts = x.reshape((-1, 1, 2))
    image = cv2.polylines(image, [pts], isClosed=True, color=(0,0,255), thickness=2)
    overlay = cv2.fillPoly(overlay, pts=[pts], color=(0,0,255))
    alpha = 0.5  # Transparency factor
    image = cv2.addWeighted(overlay, alpha, image, 1 - alpha, 0)
    # r = 600.0/image.shape[0]
    # dim = (600, int(image.shape[0]*r))

    # pts = mask[i].astype(int)
    # cv2.fillPoly(overlay, [pts], color)
    # alpha = 0.5  # Transparency factor
    # image = cv2.addWeighted(overlay, alpha, image, 1 - alpha, 0)

cv2.imshow("Predict", image)
cv2.waitKey(0)
cv2.destroyAllWindows()

PERSON [301, 37, 700, 681]
CAT [595, 319, 1012, 681]
DOG [8, 215, 331, 682]
